# 01 — Data Quality Audit

**Project:** Airbnb Market Intelligence (Data Science + Machine Learning)
**Purpose:** inspect the four supplied files, validate keys and relationships,
quantify missingness, and document the price/currency issues before any analysis.

The heavy review-file checks use Polars' lazy scanner so the 5.37m-row file is
never loaded into a spreadsheet-style structure.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)

from src.config import LISTINGS_RAW, REVIEWS_RAW, FX_SOURCE
from src.loading import load_listings
from src.build_dataset import add_price_flags

listings = add_price_flags(load_listings())
print("Listings shape:", listings.shape)
print("Files:")
for path in [LISTINGS_RAW, REVIEWS_RAW]:
    print(f"  {path.name}: {path.stat().st_size / 1e6:,.1f} MB")
print("\nFX assumption:", FX_SOURCE)
listings.head(3)

Listings shape: (279712, 62)
Files:
  Listings.csv: 158.5 MB
  Reviews.csv: 255.8 MB

FX assumption: European Central Bank euro foreign-exchange reference rates, snapshot 2026-11-11 (docs/ecb_reference_rates_2026-11-11.xml)


,listing_id,name,host_id,host_since,host_location,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_total_listings_count,host_has_profile_pic,host_identity_verified,neighbourhood,district,city,latitude,longitude,property_type,room_type,accommodates,bedrooms,amenities,price,minimum_nights,maximum_nights,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,...,instant_bookable,host_response_time_rank,currency,fx_usd_per_unit,price_usd,price_missing,price_nonpositive,price_valid,amenity_count,has_wifi,has_kitchen,has_air_conditioning,has_washer,has_heating,has_parking,has_pool,has_gym,has_elevator,has_tv,has_long_term_stay,property_family,bedrooms_missing,host_tenure_years,city_price_p01,city_price_p50,city_price_p99,price_outlier_high,price_outlier_low,price_analysable,price_above_city_median
0,281420,"Beautiful Flat in le Village Montmartre, Paris",1466919,2011-12-03,"Paris, Ile-de-France, France",<NA>,NaN,NaN,False,1.0,True,False,Buttes-Montmartre,<NA>,Paris,48.88668,2.33343,Entire apartment,Entire place,2,1.0,"[""Heating"", ""Kitchen"", ""Washer"", ""Wifi"", ""Long...",53,2,1125,100.0,10.0,10.0,10.0,10.0,...,False,NaN,EUR,1.1127,58.9731,False,False,True,5,True,True,False,True,True,False,False,False,False,False,True,apartment,False,14.940452,27.8175,89.016,667.62,False,False,True,False
1,3705183,39 mÂ² Paris (Sacre CÅ“ur),10328771,2013-11-29,"Paris, Ile-de-France, France",<NA>,NaN,NaN,False,1.0,True,True,Buttes-Montmartre,<NA>,Paris,48.88617,2.34515,Entire apartment,Entire place,2,1.0,"[""Shampoo"", ""Heating"", ""Kitchen"", ""Essentials""...",120,2,1125,100.0,10.0,10.0,10.0,10.0,...,False,NaN,EUR,1.1127,133.5240,False,False,True,8,True,True,False,True,True,False,False,False,False,False,True,apartment,False,12.950034,27.8175,89.016,667.62,False,False,True,True
2,4082273,"Lovely apartment with Terrace, 60m2",19252768,2014-07-31,"Paris, Ile-de-France, France",<NA>,NaN,NaN,False,1.0,True,False,Elysee,<NA>,Paris,48.88112,2.31712,Entire apartment,Entire place,2,1.0,"[""Heating"", ""TV"", ""Kitchen"", ""Washer"", ""Wifi"",...",89,2,1125,100.0,10.0,10.0,10.0,10.0,...,False,NaN,EUR,1.1127,99.0303,False,False,True,6,True,True,False,True,True,False,False,False,False,True,True,apartment,False,12.281999,27.8175,89.016,667.62,False,False,True,True


## 1. Primary key: Listings

In [2]:
pd.DataFrame({
    "rows": [len(listings)],
    "distinct_listing_id": [listings["listing_id"].nunique()],
    "duplicate_listing_id": [int(listings["listing_id"].duplicated().sum())],
    "null_listing_id": [int(listings["listing_id"].isna().sum())],
})

,rows,distinct_listing_id,duplicate_listing_id,null_listing_id
0,279712,279712,0,0


## 2. Missing values

In [3]:
fields = ["host_response_time", "host_response_rate", "host_acceptance_rate",
          "district", "bedrooms", "review_scores_rating", "review_scores_value", "host_since"]
missing = pd.DataFrame({
    "missing_count": listings[fields].isna().sum(),
    "missing_pct": (100 * listings[fields].isna().mean()).round(2),
})
missing.sort_values("missing_pct", ascending=False)

,missing_count,missing_pct
district,242700,86.77
host_response_time,128782,46.04
host_response_rate,128782,46.04
host_acceptance_rate,113087,40.43
review_scores_value,91785,32.81
review_scores_rating,91405,32.68
bedrooms,29435,10.52
host_since,165,0.06


In [4]:
(listings.groupby("city", observed=True)[["district", "bedrooms", "review_scores_rating"]]
         .apply(lambda g: 100 * g.isna().mean()).round(1))

,district,bedrooms,review_scores_rating
city,,,
Paris,100.0,20.7,25.7
New York,0.0,9.7,27.7
Sydney,100.0,6.6,33.5
Rome,100.0,3.2,24.4
Rio de Janeiro,100.0,6.6,39.4
Istanbul,100.0,8.3,54.2
Mexico City,100.0,4.0,27.8
Bangkok,100.0,11.1,42.2
Cape Town,100.0,7.2,29.6


`district` is populated only for New York. Review scores exist only for listings
that have been reviewed. Neither gap is missing at random, so every later
statistic states which sub-population it describes.

## 3. Price validity and currency structure

In [5]:
pd.DataFrame({
    "check": ["price missing", "price == 0", "price < 0",
              "above per-city 99th pct (USD)", "below per-city 1st pct (USD)"],
    "rows": [int(listings["price"].isna().sum()),
             int((listings["price"] == 0).sum()),
             int((listings["price"] < 0).sum()),
             int(listings["price_outlier_high"].sum()),
             int(listings["price_outlier_low"].sum())],
})

,check,rows
0,price missing,0
1,price == 0,113
2,price < 0,0
3,above per-city 99th pct (USD),2753
4,below per-city 1st pct (USD),2454


In [6]:
currency_ref = (listings.groupby("city", observed=True)
                        .agg(currency=("currency", "first"),
                             listings=("listing_id", "size"),
                             usd_per_unit=("fx_usd_per_unit", "first"))
                        .reset_index())
currency_ref

,city,currency,listings,usd_per_unit
0,Paris,EUR,64690,1.112700
1,New York,USD,37012,1.000000
2,Sydney,AUD,33630,0.685920
3,Rome,EUR,27647,1.112700
4,Rio de Janeiro,BRL,26615,0.179970
5,Istanbul,TRY,24519,0.019909
6,Mexico City,MXN,20065,0.053592
7,Bangkok,THB,19361,0.029682
8,Cape Town,ZAR,19086,0.060372
9,Hong Kong,HKD,7087,0.127460


In [7]:
(listings[listings["price_valid"]]
    .groupby("city", observed=True)
    .agg(currency=("currency", "first"),
         n=("listing_id", "size"),
         median_local=("price", "median"),
         p99_local=("price", lambda s: s.quantile(0.99)),
         max_local=("price", "max"),
         median_usd=("price_usd", "median"))
    .round(2))

,currency,n,median_local,p99_local,max_local,median_usd
city,,,,,,
Paris,EUR,64628,80.0,600.00,12000,89.02
New York,USD,36984,99.0,850.00,10000,99.00
Sydney,AUD,33630,120.0,1721.39,28613,82.31
Rome,EUR,27641,65.0,600.00,10571,72.33
Rio de Janeiro,BRL,26608,280.0,5696.64,625216,50.39
Istanbul,TRY,24517,252.0,3805.88,179532,5.02
Mexico City,MXN,20058,664.0,7294.72,499000,35.59
Bangkok,THB,19360,1100.0,15553.39,300177,32.65
Cape Town,ZAR,19086,1069.0,22018.90,180000,64.54


Every price statistic above is computed **inside** a city. The local median and
the USD median are different quantities and are never compared as if they were
the same currency. The huge local maxima (Rio de Janeiro, Istanbul, Bangkok) are
implausible nightly rates and are handled as extreme observations.

## 4. Reviews: keys, duplicates and join coverage

In [8]:
from src.reviews import validate_reviews

findings = validate_reviews(REVIEWS_RAW)
pd.Series(findings).to_frame("value")

,value
rows,5373143
unique_listings,193556
unique_review_ids,5372983
unique_reviewers,4450005
min_date,2008-11-16
max_date,2021-03-01
null_listing_id,0
null_review_id,0
null_date,0
null_reviewer_id,0


In [9]:
from src.reviews import duplicate_review_evidence

dup = duplicate_review_evidence(REVIEWS_RAW)
print(f"Rows involved in repeated review_id values: {len(dup):,}")
print(f"Distinct repeated review_id values: {dup['review_id'].nunique()}")
dup.head(10)

Rows involved in repeated review_id values: 320


Distinct repeated review_id values: 160


,listing_id,review_id,date,reviewer_id
0,2997346,200088015,2017-10-04,123346683
1,22219088,200088015,2017-10-04,123346683
2,2997346,205182329,2017-10-21,23770481
3,22219088,205182329,2017-10-21,23770481
4,2997346,223954194,2018-01-02,77129337
5,22219088,223954194,2018-01-02,77129337
6,2997346,256197483,2018-04-22,115838328
7,22219088,256197483,2018-04-22,115838328
8,466155,256940062,2018-04-24,172857558
9,22219250,256940062,2018-04-24,172857558


The duplicated `review_id` values are real in the supplied file. Of the repeated
IDs, some carry identical payloads and some conflict. Because
`(listing_id, review_id)` is unique, the reviews are aggregated by `listing_id`
and `review_id` is never used as a primary key. No review rows are deleted, so
the 5.37m review records remain fully accounted for.

## 5. Categorical structure

In [10]:
display(listings["room_type"].value_counts().to_frame("listings"))
display(listings["property_family"].value_counts().to_frame("listings"))
print("Raw property_type values:", listings["property_type"].nunique())

,listings
room_type,
Entire place,182005
Private room,86988
Hotel room,5857
Shared room,4862


,listings
property_family,
apartment,195403
house,33197
condominium,15911
hotel,9024
guesthouse,6200
loft,5598
bed and breakfast,5496
hostel,2638
villa,2100


Raw property_type values: 144


## 6. Decisions and limitations

1. `listing_id` is the Listings primary key; `(listing_id, review_id)` is the
   Reviews unique key.
2. Reviews are left-joined to listings: no review is a real zero, not a missing row.
3. Prices are local currency; USD conversion uses one documented ECB snapshot.
4. Non-positive prices are invalid. Values above each city's 99th percentile are
   flagged as extreme and excluded from model training only.
5. No review text exists, so sentiment analysis is out of scope.
6. Advertised price is not revenue, occupancy or profit.

The full generated report is `reports/data_quality_report.md`.